# Weak Values & Weak Measurements
## The physics
Typically, quantum practitioners are familiar with the concept of measuring: in order to obtain information from a system one needs to perform a measurement. However, measuring implies collapsing the wave function into the eigenstate of the measured eigenvalue, losing all previous information about the original state.

Nevertheless, not so many people know there are different types of measurements: the previously described *projective* measurements and the less known *weak* measurements.

**Weak measurements** are a special type of measurement in which the interaction with the system is so small that the state only partially collapses. To do so, one needs to entangle the original system with a a pointer's system (the ancilla/auxiliary system) that will react and measure the change suffered by the original system. Namely, the total initial state is
$$
|\psi_s \rangle \otimes |\phi \rangle,
$$
where $|\psi_s\rangle$ is the original system and $|\phi \rangle$ is the auxiliary one (this state is chosen to be a Gaussian state to represent the minimum perturbation to the system).
One key feature of weak measurements is that, due to the extremely small interaction, the uncertainty for each measurement is really big and thus, many measurements need to be taken in order to obtain a reasonable weak value.

The Hamiltonian controlling the interaction of such a system is a von Neumann-type interaction Hamiltonian,
$$
\mathcal{H}_t=\tilde{g}(t) \hat{A} \otimes \hat{p}.
$$
This Hamiltonian is defined by $\tilde{g}(t)$, the interaction strength over time, $\hat{A}$, the observable from which we want to extract information from the original system. And $\hat{p}$ the pointer's momentum operator (the one that control's how much the pointer shifts). Since $\hat{A}, \hat{p}$ do not have an explicit dependence on time, and assuming the interaction strength to be really small compared to the uncertainty of each measurement (which allows $g=\int_0^t \tilde{g}(t) dt$), one can neglect the dependence on time as
$$
\mathcal{H}_t=g \hat{A} \otimes \hat{p}.
$$

The total state with the interaction Hamiltonian now becomes
$$
|\Phi \rangle = e^{-i g \hat{A} \otimes \hat{p}}|\psi_s\rangle \otimes |\phi\rangle.
$$

Lastly, it is necessary to introduce the last step of weak measurements, the one from which the magic arises: the **post-selection**. As would be the case for a projective measurement, it is necessary to identify which value would the measurement return for a given final state. In the case of weak measurements, post-selecting gives rise to quite an impressive phenomenon:
$$
|\phi_f \rangle =(\langle \psi_f| \otimes \mathbb{I}) e^{-i g \hat{A} \otimes \hat{p}}|\psi_s\rangle \otimes |\phi\rangle
$$
(after turning the exponential into a series expansion)
$$
\approx \langle \psi_f | \psi_s \rangle (1-ig \frac{\langle \psi_f |\hat{A} | \psi_s \rangle}{\langle \psi_f | \psi_s \rangle}\hat{p})|\phi\rangle.
$$
Here, $\frac{\langle \psi_f |\hat{A} | \psi_s \rangle}{\langle \psi_f | \psi_s \rangle}=A_w$ represents the **Weak Value of operator $\hat{A}$**.
From here, and after some algebra, one can derive the following result:
$$
\langle q \rangle = g \text{Re}(A_w)
$$
where $\langle q \rangle$ represents the final position of the meter's pointer.
An extremely important property of weak values is that **they can fall far outside the spectrum of their operator** i.e. $A_w$ can be a value which is not an eigenvalue of $\hat{A}$. In fact, they can be extremely big, depending on how close to being "orthogonal" the pre and post selected states are the denominator will become smaller.

## The program

Inside a PennyLane circuit, the interaction Hamiltonian is defined differently than in theory:
$$
H=A\otimes Y
$$
where $A$ is still the operator we want to obtain the weak value from and $Y$ is Pauli Y matrix. This way also allows to codify the information from A inside the auxiliary system without having to prepare the Gaussian state, it will start in state $|0\rangle$.
We will define the original system as wire 0 and the auxiliary system as wire 1.
The evolution operator is thus
$$
U=e^{-i \theta H}\approx \mathbb{I}-i\theta A\otimes Y,
$$
for small $\theta$.
The total initial state is therefore
$$
|\Phi \rangle\approx   (\mathbb{I}-i\theta A\otimes Y)|\psi_i\rangle\otimes \ket{0}=|\psi_i\rangle\otimes|0\rangle-i\theta A|\psi_i\rangle\otimes Y |0\rangle
$$
$$
=|\psi_i\rangle\otimes|0\rangle-i\theta A|\psi_i\rangle\otimes i |1\rangle=|\psi_i\rangle\otimes\ket{0}+\theta A|\psi_i\rangle\otimes|1\rangle.
$$
Applying the post selection:
$$
|\phi_f\rangle=(\langle \psi_f | \otimes \mathbb{I})(|\psi_i\rangle\otimes\ket{0}+\theta A|\psi_i\rangle\otimes|1\rangle)
$$
$$
=\langle \psi_f|\psi_i\rangle (|0\rangle+\theta A_w |1\rangle).
$$
Once normalized:
$$
|\phi_f\rangle = \frac{1}{\sqrt{1+\theta^2|A_w|^2}}(|0\rangle+\theta A_w |1\rangle).
$$
Now, one would need to compute the density matrix (this is left as an exercise for the reader) in order to finally achieve
$$
\langle X \rangle = \text{Tr}(\rho \sigma_x)=\frac{2\theta \text{Re}(A_w)}{1+\theta^2|A_w|^2}.
$$
Please, note that PennyLane will automatically normalize the state so there is no need to do so explicitly in the program.
The same is true for the imaginary part:
$$
\langle Y \rangle = \text{Tr}(\rho \sigma_y)=\frac{2\theta \text{Im}(A_w)}{1+\theta^2|A_w|^2}.
$$

In [13]:
import pennylane as qp
from pennylane import numpy as np

dev=qp.device("default.qubit",wires=2)


@qp.qnode(dev,shots=500000)
def circuit_weak_value(A_matrix,theta, initial_state_vector, final_state_vector,measurement_basis="X"):

    A_op=qp.Hermitian(A_matrix,wires=0)

    # Weak interaction Hamiltonian
    H=A_op @ qp.PauliY(wires=1)

    #Set the initial state of the system
    qp.StatePrep(initial_state_vector,wires=0)

    # Evolution caused by the weak interaction
    qp.exp(H, coeff=-1j * theta)

    # Post-selection (only on system wire)
    qp.adjoint(qp.StatePrep(final_state_vector,wires=0))

    # Selection of the basis on which we want to measure the ancilla (wire 1)
    # For the real part
    if measurement_basis=="X":
        qp.Hadamard(wires=1)
    # For the imaginary part
    if measurement_basis=="Y":
        qp.adjoint(qp.S(wires=1))
        qp.Hadamard(wires=1)

    return qp.counts(wires=[0, 1])


The previous code simulates the behaviour of the weak measurement, but understanding what it returns is not trivial.
The function returns a dictionary of values following the structure "state:times measured" for states 00, 01, 10, 11. Where the first digit is the system wire and the second the auxiliary one.
### What does it mean if the system wire is 0?:
In PennyLane the state in which a qubit is initialized is always 0. At the beginning of the function we change that state to some $|\psi_i\rangle$. Then, applying the weak interaction evolution we slightly change that state into some unknown state. However, we wish to measure some value, and thus one needs to choose a post-selection: the state over which we want to project our system state. Once we measure, PennyLane will always give two possible values, either 0 or 1 for that qubit. 0 always implies your post-selection was successful: if we began at |0>, apply some gate $U$, then a post-selection and then we measure and the circuit returns 0, that means our post-selection was $U^\dagger$ i.e. after the original $U$ your state ended up in precisely the state you wanted to measure, and so it is a valid post-selection.

Therefore, we need to keep the results of the dictionary for which the system wire was 0: the results in which the post-selection was successful.

Now, for the second digit, the auxiliary/ancilla qubit: it also can take either 0 or 1 as its value. This basically represents the eigenstate on which the auxiliary system collapsed. For example, if we choose the measuring basis as X, we apply a Hadamard gate to the original state $|0\rangle$ of the auxiliary wire: physically, this behaves as rotating the Bloch sphere so that the new measuring axis is X (PennyLane always measures in Z direction, which is now inhabited by the X axis because of the rotation induced by Hadamard gate). Thus, if the result is 0 that means the state collapsed in $|+\rangle$ and if it is 1 it collapsed in $|-\rangle$, each with eigenvalues +1 and -1 respectively over X basis.

The expectation value is quite simple to compute from this logic:
$$
\langle X \rangle = \frac{+1\cdot \text{times the system collapsed on}|+\rangle + (-1)\cdot\text{times the system collapsed on}|-\rangle}{\text{total number of measurements with correct post-selection}}.
$$
Same applies for the expectation value of Y, just with a different roation.
Therefore, the last part of the program is focused on analyzing these results and computing the final weak value.



In [14]:
def expectation_values(counts,theta):

    n_0=0 # Number of times eigenvalue +1 appears
    n_1=0 # Number of times eigenvalue -1 appears

    for state ,frequency in counts.items():
        system_state = state[0]
        ancilla_state = state[1]

        if system_state=="0":

            if ancilla_state=="0":
                n_0+=frequency
            if ancilla_state=="1":
                n_1+=frequency
                
    total_frequency = n_0+n_1

    if total_frequency==0.0:
        return 0.0

    exp_val=((+1*n_0)+(-1*n_1))/total_frequency

    return exp_val

The program is now complete. To test it, one could use the following values
$$
A=\frac{\sigma_x+\sigma_y}{\sqrt{2}} \quad , \quad |\psi_i\rangle=|+\rangle \quad , \quad |\psi_f\rangle=|y+\rangle.
$$
The Weak Value is
$$
A_w=\frac{\langle \psi_f |A|\psi_i\rangle}{\langle \psi_f | \psi_i \rangle}=\sqrt{2}\approx 1.4142.
$$
Please, note the Weak Value obtained by the simulation will not be exactly $A_w$ due to the innate uncertainty of these measurements. To reduce the uncertainty one must increase the number of shots, since the statistical error scales as $\frac{1}{\sqrt{N}}$.

In [15]:


theta= 0.05 # Needs to be small for the series expansion required as part of the weak interaction

# Definition of the observable operator
A_matrix = np.array([
    [0, (1 - 1j) / np.sqrt(2)],
    [(1 + 1j) / np.sqrt(2), 0]
], dtype=complex)

# The initial system state set to be |+>
psi_initial = np.array([1 / np.sqrt(2), 1 / np.sqrt(2)], dtype=complex)

# The system post-selection state set to be |-y>
psi_final = np.array([1 / np.sqrt(2), 1j / np.sqrt(2)], dtype=complex)

circuit_results_X=circuit_weak_value(A_matrix, theta, psi_initial, psi_final, measurement_basis="X")
circuit_results_Y=circuit_weak_value(A_matrix, theta, psi_initial, psi_final, measurement_basis="Y")

exp_val_X=expectation_values(circuit_results_X,theta)
exp_val_Y=expectation_values(circuit_results_Y,theta)

real_part=exp_val_X/(2*theta)
imaginary_part=exp_val_Y/(2*theta)

print("The Weak Value is:\t", real_part , "+", imaginary_part, "i")

The Weak Value is:	 1.4315821405477898 + 0.009898223907403712 i
